# Single-gene research: upstream → receptors, downstream → gene expression, synthetic lethality

**Input: one candidate gene** (`GENE`). Output: ranked receptors (upstream alternatives), signed TF/target-gene layer (what the gene does), and DepMap synthetic-lethal partners (parallel targets).

Data: SIGNOR + Reactome (signaling, signed), Lambert TF list, DoRothEA A–C (TF→gene), UniProt (receptors, location), DepMap (genetics). Set `DEMO = True` for the offline curated AKT1 axis.

## 1. Parameters
Edit only this cell. `DEPMAP_DIR` is optional (needed for co-essentiality and synthetic lethality; see `fetch_depmap.py`). `TISSUE` filters the graph to genes expressed in an HPA tissue (e.g. `"liver"`); leave `None` for no filter.

Database locations come from the YAML config (`config.example.yaml` → copy to `target_net.yaml`); parameters here only override it.

In [ ]:
GENE = "AKT1"          # candidate gene (HGNC symbol)
DEMO = False           # False = your databases from target_net.yaml; True = offline curated demo graph
CONFIG = None          # None = find target_net.yaml in this folder or any parent (project root), else ~/.target_net.yaml; or give a path
DEPMAP_DIR = None      # overrides databases.depmap.dir in the YAML      # e.g. "~/data/depmap"
TISSUE = None          # e.g. "liver"
MAX_DEPTH = MIN_SCORE = HUB_PENALTY = None   # None = values from the YAML (walk: section)
TARGET_NET_DIR = ".."   # folder containing target_net.py (this notebook lives in target_net/notebooks/)
FOCUS = None           # genes to highlight; everything else is drawn as small grey circles. Paste plain text, e.g.
# FOCUS = """
# GSK3B -1.8
# FOXO3 -1.2
# MTOR 0.9
# CREB1
# """                   # names separated by newlines/commas/spaces; a number after a name = value (colour blue→red). A file path or a Python list/dict also works.

## 2. Setup
Make the package importable and move into a working folder (`runs/`) so outputs do not mix with the code.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
sys.path.insert(0, os.path.abspath(TARGET_NET_DIR))
from targetnet.api import research_gene, summarize_gene, show, figure, legend, gene_table, focus_summary
from targetnet.plots import use_cdn_renderer
use_cdn_renderer()   # plotly.js from CDN -> small .ipynb files
import pandas as pd
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 160)
OUT_ROOT = os.path.join(TARGET_NET_DIR, "runs")   # each analysis gets runs/<YYYYMMDD_HHMMSS>_<mode>_<genes>/

### Key to the figures
Every figure carries its own caption (lines, arrowheads, colours). `legend(kind)` prints the same key as text — kinds: `flow`, `cell_map`, `crosstalk`, `volcano`.

In [ ]:
legend("flow")

## 3. Run the three analyses
Upstream walk stops at receptors; downstream walk stops at transcription factors and adds DoRothEA targets; synthetic lethality runs only when DepMap is available (or in demo mode, where it is **simulated**).

In [ ]:
r = research_gene(GENE, config=CONFIG, demo=DEMO, depmap=os.path.expanduser(DEPMAP_DIR) if DEPMAP_DIR else None, tissue=TISSUE,
                  max_depth=MAX_DEPTH, min_score=MIN_SCORE, hub_penalty=HUB_PENALTY, out=OUT_ROOT)
print(summarize_gene(r))
print("outputs:", r["run_dir"])

## 4. All genes in one table
`r["genes"]` (also saved as `gene_table.csv` in the run folder): one row per gene reached from the seed, upstream and/or downstream.
- `direction` upstream / downstream / both · `hops` shortest distance (either direction), `hops_up`, `hops_down`
- `role` receptor / TF / intermediate · `score_*` best single path · `influence_*` sum over all paths (ranking) · `influence_total`
- `sign_up` = the gene's net effect **on** the seed (+1 activates, −1 inhibits, 0 mixed) · `sign_down` = the seed's net effect **on** the gene; `conf_*` = sign confidence
- `feedback` on a loop through the seed · `coessentiality_r` (DepMap) · `physical_with_seed` · `compartment` · `n_target_genes` (TFs) · `sl_hit` · best paths

In [ ]:
genes = gene_table(r, focus=FOCUS)      # with FOCUS set: focus genes first, columns focus / focus_value
print(len(genes), "genes;", genes.direction.value_counts().to_dict(), "| roles:", genes.role.value_counts().to_dict())
if FOCUS is not None: print(focus_summary(r, FOCUS)); genes.to_csv(os.path.join(r["run_dir"], "gene_table_focus.csv"), index=False)
genes.head(30)

In [ ]:
# filter examples: receptors within 3 hops that activate the seed, or TFs the seed represses
genes.query("role == 'receptor' and hops <= 3 and sign_up == 1").head(15)
# genes.query("role == 'TF' and sign_down == -1").head(15)

## 5. Upstream: receptors that feed the gene
Columns: `hops`, `score` (best single path: confidence × hub penalty), `influence` (sum over **all** paths up to max depth — better for ranking), `sign` (consensus over all paths: +1 activates the gene, −1 inhibits, 0 mixed), `sign_confidence` (|act−inh| / total), `coessentiality_r` (DepMap), `best_path`.

In [ ]:
r["receptors"].sort_values("influence", ascending=False)[["node","hops","score","influence","sign","sign_confidence","coessentiality_r","physical_with_seed","best_path"]].head(20)

In [ ]:
figure(r["views"]["upstream_flow"], focus=FOCUS).show()          # Plotly, inline. Full D3 view: show(r["views"]["upstream_flow"])

## 6. Downstream: transcription factors and target genes
`sign` on a TF = consensus net effect of the gene on that TF over all paths (`sign_bestpath` keeps the single-best-path sign); on a target gene = TF sign × mode of regulation. Nodes with `feedback = True` are also upstream of the gene (loop).

In [ ]:
r["tfs"].sort_values("influence", ascending=False)[["node","hops","score","influence","sign","sign_confidence","feedback","best_path"]].head(20)

In [ ]:
tg = r["target_genes"]
(tg.groupby("tf").agg(n_genes=("gene","nunique"), up=("sign", lambda s:(s==1).sum()), down=("sign", lambda s:(s==-1).sum()))
   .sort_values("n_genes", ascending=False)) if len(tg) else "no regulon layer"

In [ ]:
figure(r["views"]["downstream_flow"], focus=FOCUS).show()          # Plotly, inline. Full D3 view: show(r["views"]["downstream_flow"])

## 7. Synthetic lethality (separate view)
Genes significantly more essential when the candidate gene is lost (Welch t-test, BH-FDR < 0.05, Δ < −0.15). `on_pathway = True` means the hit is already in the signaling neighbourhood (not a parallel target).

In [ ]:
sl = r["sl"]
sl[sl.significant].sort_values("delta")[["gene","delta","padj","n_lost","on_pathway","coessentiality_r"]].head(20) if len(sl) else ("SL not run: " + r.get("sl_note", "no DepMap configured"))

In [ ]:
figure(r["views"]["sl"]).show()          # Plotly, inline. Full D3 view: show(r["views"]["sl"])

## 8. Location view
Same nodes placed by subcellular compartment (UniProt).

In [ ]:
figure(r["views"]["downstream_cell_map"], focus=FOCUS).show()          # Plotly, inline. Full D3 view: show(r["views"]["downstream_cell_map"])

## 9. Interpretation checklist
- Upstream alternatives: receptors at 2–3 hops with positive sign and (if DepMap) positive co-essentiality.
- Downstream: which TF programmes does the gene switch on/off? Are endpoints tissue-relevant (use `TISSUE`)?
- Synthetic lethal: hits with `on_pathway = False` are the parallel-pathway candidates.
- Sanity: does the result recover the textbook biology for this gene? If not, tune `MIN_SCORE` / `HUB_PENALTY` before trusting novel hits.